# Тема 11. Протоколы межагентного взаимодействия: контракты сообщений и MCP vs A2A

**Модуль III · Пример 4 из 5**

Два разных протокола образуют экосистему распределённой агентной системы:

- **MCP (Model Context Protocol)** — взаимодействие **«агент ↔ инструмент»** (вызов функций/данных). Аналогия: универсальный разъём к инструментам.
- **A2A (Agent-to-Agent)** — взаимодействие **«агент ↔ агент»** (обмен сообщениями между ролями). Аналогия: прямая связь между устройствами.

Их **не следует смешивать**: вызов инструмента и сообщение другому агенту — разные сущности. В этой тетради задаём строгий **контракт межагентного сообщения** и показываем оба вида обмена раздельно.

In [1]:
import os, json
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal, Any
import mas_domain as dom

BASE_URL = os.environ.get("LLM_BASE_URL", "http://localhost:8000/v1")
API_KEY  = os.environ.get("LLM_API_KEY", "")
MODEL    = os.environ.get("LLM_MODEL", "qwen/qwen3.7-max")
client = OpenAI(api_key=API_KEY or "EMPTY", base_url=BASE_URL)
print("Готово:", MODEL)

Готово: qwen/qwen3.7-max


## Контракт межагентного сообщения (A2A)

Строгий типизированный контракт: идентификация отправителя и получателя, идентификатор подзадачи, **статус результата** и полезная нагрузка. Валидация через Pydantic делает обмен между ролями надёжным и трассируемым.

In [2]:
class AgentMessage(BaseModel):
    sender: str                                        # кто отправил
    recipient: str                                     # кому
    subtask_id: str                                    # к какой подзадаче относится
    status: Literal["ok", "failed", "needs_input"]     # статус результата
    payload: dict[str, Any] = Field(default_factory=dict)
    notes: str = ""

# Пример корректного сообщения от специалиста координатору
msg = AgentMessage(sender="orders_agent", recipient="coordinator",
                   subtask_id="s1", status="ok",
                   payload={"order_id": "ORD-1002", "state": "in_transit", "eta_days": 2})
print(msg.model_dump_json(indent=2))

{
  "sender": "orders_agent",
  "recipient": "coordinator",
  "subtask_id": "s1",
  "status": "ok",
  "payload": {
    "order_id": "ORD-1002",
    "state": "in_transit",
    "eta_days": 2
  },
  "notes": ""
}


## MCP-обмен (агент ↔ инструмент)

Здесь агент вызывает **инструмент** — это MCP-уровень (в нашем примере инструменты локальны, но семантика та же: типизированный вызов внешней возможности). Результат инструмента — не межагентное сообщение.

In [3]:
def agent_calls_tool(user_query: str) -> dict:
    r = client.chat.completions.create(model=MODEL, max_tokens=300,
        tools=dom.tools_subset(["get_order", "check_shipping"]),
        messages=[{"role": "user", "content": user_query}])
    tc = r.choices[0].message.tool_calls
    if not tc:
        return {"tool": None, "result": r.choices[0].message.content}
    name, args = tc[0].function.name, json.loads(tc[0].function.arguments)
    result = dom.run_tool(name, args)          # <-- MCP-уровень: агент -> инструмент
    print(f"MCP-вызов: {name}({args}) -> {json.dumps(result, ensure_ascii=False)[:120]}")
    return {"tool": name, "args": args, "result": result}

tool_out = agent_calls_tool("Проверь доставку заказа ORD-1002")

MCP-вызов: check_shipping({'order_id': 'ORD-1002'}) -> {"order_id": "ORD-1002", "carrier": "CDEK", "state": "in_transit", "eta_days": 2}


## A2A-обмен (агент ↔ агент)

Теперь специалист **упаковывает** результат инструмента в межагентное сообщение по контракту и **передаёт** его координатору. Координатор принимает **типизированное** сообщение (не сырой вызов инструмента) — это A2A-уровень.

In [4]:
def orders_agent_reply(subtask_id: str, tool_result: dict) -> AgentMessage:
    status = "ok" if tool_result.get("state") not in (None, "no_tracking") else "needs_input"
    return AgentMessage(sender="orders_agent", recipient="coordinator",
                        subtask_id=subtask_id, status=status, payload=tool_result,
                        notes="статус доставки получен")

def coordinator_receive(messages: list[AgentMessage]) -> str:
    # Координатор синтезирует ответ ТОЛЬКО из типизированных сообщений A2A
    joined = "\n".join(f"[{m.sender}->{m.recipient} | {m.status}] {json.dumps(m.payload, ensure_ascii=False)}"
                       for m in messages)
    r = client.chat.completions.create(model=MODEL, max_tokens=300,
        messages=[{"role": "system", "content": "Ты координатор. Составь ответ клиенту по сообщениям агентов."},
                  {"role": "user", "content": joined}])
    return r.choices[0].message.content

a2a_msg = orders_agent_reply("s1", tool_out["result"])
print("A2A-сообщение:\n", a2a_msg.model_dump_json(indent=2))
print("\nОтвет координатора:\n", coordinator_receive([a2a_msg])[:300])

A2A-сообщение:
 {
  "sender": "orders_agent",
  "recipient": "coordinator",
  "subtask_id": "s1",
  "status": "ok",
  "payload": {
    "order_id": "ORD-1002",
    "carrier": "CDEK",
    "state": "in_transit",
    "eta_days": 2
  },
  "notes": "статус доставки получен"
}



Ответ координатора:
 Добрый день! 

Ваш заказ **ORD-1002** уже передан в службу доставки **СДЭК** и сейчас находится в пути. 📦

Ориентировочный срок доставки — **в течение 2 дней**. 

Вы можете отслеживать перемещение посылки на официальном сайте или в приложении СДЭК. Если у вас возникнут дополнительные вопросы или пон


## Разграничение протоколов — почему их нельзя смешивать

| Обмен | Протокол | Что передаётся | Пример из тетради |
|-------|----------|----------------|-------------------|
| агент → инструмент | **MCP** | вызов функции + типизированные аргументы | `get_order`, `check_shipping` |
| агент → агент | **A2A** | сообщение по контракту (отправитель, статус, payload) | `AgentMessage(...)` |

Смешение (например, передача сырого результата инструмента напрямую как «сообщения агента» без контракта, статуса и идентификации отправителя) ломает трассируемость и обработку ошибок: получатель не знает, кто прислал, в каком статусе и к какой подзадаче относится результат.

### Мультимодальные роли (факультативно)
Мультимодальный агент — это ещё одна **специализированная роль** в той же схеме: агент компьютерного зрения (VLM — анализ изображений/интерфейсов) или голосовой агент (речь→текст→бизнес-логика→синтез). Он подключается к координатору **тем же контрактом A2A**, а к своей модели/инструменту — по MCP-семантике. Ключевое ограничение голосовых агентов — задержка (порог естественности диалога ~300 мс).

## Итоги

- **A2A-контракт** (типизированное сообщение: отправитель, получатель, подзадача, статус, payload) делает межагентный обмен надёжным и трассируемым.
- **MCP** (агент↔инструмент) и **A2A** (агент↔агент) — разные уровни; их разделение обязательно для отладки и обработки ошибок.
- Структурированный вывод (Pydantic) — общий формат обмена и на уровне инструментов, и между ролями.
- Мультимодальные агенты встраиваются как роли по тем же протоколам.

**Дальше:** Темы 10/12 — бенчмарк архитектур и дифференцированный подбор моделей под роли.